# Entregável 1 — Especificação e Baseline

> **Grupo:** Mariana Aparecida Ferreira, Vinicius Luis Belem Bronzatti
> **Tema/Projeto:** RiskOps — Sistema Multiagente para Gestão Automatizada do Ciclo de Vida de Regras de Risco
> **Data:** 03/09/2026

Este notebook contém a especificação inicial do sistema, a implementação de um baseline
funcional, casos de teste e uma análise crítica das limitações observadas.


# 1. Descrição do problema

Sistemas de prevenção a fraude em produtos financeiros (ex.: abertura de conta) costumam
depender de **regras de decisão** (condições sobre atributos da transação/aplicação) para
sinalizar casos suspeitos. Com o tempo, o comportamento dos fraudadores muda e as regras
"envelhecem": perdem poder de detecção, passam a gerar excesso de falsos positivos, ou
simplesmente nunca foram validadas com rigor estatístico ao serem criadas. Na prática,
raramente existe um processo sistemático para revisar essas regras — elas ficam ativas até
alguém notar um problema grande o suficiente para investigar manualmente.

O RiskOps, projeto de referência deste grupo, propõe um sistema multiagente completo para
gerir esse ciclo de vida (monitoramento contínuo, diagnóstico, geração de regras candidatas,
backtest, revisão e publicação — ver arquitetura-alvo no `README.md` do repositório). **Esse
sistema completo não é o objetivo deste Entregável 1.** Aqui tratamos apenas a fatia mais
simples e fundamental desse problema: dado **uma única regra de risco já existente**, produzir
um parecer, fundamentado em métricas de backtest sobre dados históricos rotulados, sobre se
essa regra ainda vale a pena ser mantida.

Isso é relevante porque é exatamente o julgamento que hoje um analista de risco faz
manualmente (e raramente faz com a frequência necessária): olhar uma regra, cruzar com dados
históricos, e decidir se ela continua útil.


# 2. Usuário-alvo e stakeholders

- **Usuário principal:** analista de risco/fraude responsável por manter um portfólio de
  regras de decisão.
- **Usuários secundários:** gerente de risco, que aprova mudanças em regras; time de dados,
  que constrói e testa novas regras candidatas.
- **Stakeholders relevantes:** compliance/auditoria (precisa de rastreabilidade das decisões
  sobre regras); produto/negócio (sofre com fricção de falsos positivos em clientes
  legítimos); engenharia (mantém o motor de regras em produção).
- **Objetivos desses usuários:** obter, rapidamente e com evidência quantitativa, uma leitura
  sobre se uma regra específica ainda está cumprindo seu papel — sem precisar rodar
  manualmente um backtest e interpretar os números a cada vez.


# 3. Casos de uso principais

**UC1 — Revisar uma regra ativa**
- **Ator:** analista de risco
- **Entrada:** `rule_id` de uma regra já cadastrada e ativa no registro de regras
- **Objetivo:** saber se a regra ainda deve continuar ativa
- **Saída esperada:** veredito (manter/revisar/aposentar) + métricas + justificativa
- **Situação de sucesso:** veredito condizente com as métricas reais, com latência aceitável

**UC2 — Avaliar uma regra candidata antes de promovê-la**
- **Ator:** analista de risco
- **Entrada:** `rule_id` de uma regra com status "candidate" no registro
- **Objetivo:** decidir se a candidata é boa o suficiente para virar regra ativa
- **Saída esperada:** mesmo formato do UC1
- **Situação de sucesso:** idem UC1

**UC3 — Consultar uma regra inexistente ou entrada inválida**
- **Ator:** analista de risco (ou erro de digitação/integração)
- **Entrada:** `rule_id` que não existe no registro, ou vazio
- **Objetivo:** receber um retorno compreensível em vez de um erro técnico
- **Saída esperada:** mensagem de erro clara, sem *stack trace*
- **Situação de sucesso:** o sistema não quebra e explica o que houve


# 4. Escopo, não-objetivos e premissas

| | |
|---|---|
| **Escopo** | diagnóstico de **uma** regra de risco por vez, contra dados históricos rotulados, produzindo um parecer estruturado (veredito + justificativa + sugestão) |
| **Não-objetivos** | monitoramento automático de todo o portfólio de regras; geração automática de regras candidatas; fluxo de aprovação humana; publicação/versionamento de mudanças (o motor determinístico já suporta isso, mas não é orquestrado por este baseline); memória de longo prazo entre execuções; múltiplos agentes/orquestração |
| **Premissas** | (i) a regra já existe no registro versionado (schema e `RuleStore` construídos na Fase 1 do projeto); (ii) os dados usados são uma **amostra estratificada de 20.000 linhas** do dataset Bank Account Fraud (BAF, NeurIPS 2022), preservando a taxa de fraude original (~1,1%) — decisão para manter o notebook 100% executável sem exigir credenciais do Kaggle; a base completa (1.000.000 de linhas) já está integrada ao motor de regras do projeto e é usada fora deste notebook; (iii) o rótulo `fraud_bool` é confiável, conforme o datasheet oficial do dataset |


# 5. Entradas e saídas

## Entradas
Um `rule_id` (string), referenciando uma regra já registrada no `rule_registry` do projeto
(construído na Fase 1). Implicitamente, o sistema também usa o dataset de transações
históricas rotuladas (amostra do BAF, carregada uma vez no início do notebook).

## Saídas
Um parecer estruturado contendo:
- `veredito`: `"manter"`, `"revisar"` ou `"aposentar"`;
- `justificativa`: texto citando as métricas calculadas;
- `sugestao`: próximo passo concreto sugerido;
- métricas de backtest: `detection_rate`, `false_positive_rate`, `precision`, `approval_rate`;
- metadados de instrumentação: latência, número de chamadas ao modelo, tokens de entrada/saída.


# 6. Requisitos funcionais

- **RF-01:** dado um `rule_id` válido, o sistema calcula `detection_rate`,
  `false_positive_rate`, `precision` e `approval_rate` através do motor de backtest
  determinístico da Fase 1 — verificável comparando o resultado com uma chamada manual da
  mesma função de backtest sobre os mesmos dados (igualdade exata).
- **RF-02:** o parecer em linguagem natural cita os valores numéricos calculados (taxa de
  detecção e taxa de falso positivo, arredondados) — verificável por busca desses números
  (com tolerância de arredondamento) no texto da justificativa.
- **RF-03:** o veredito categórico do modelo concorda com um veredito de referência calculado
  deterministicamente (limiares sobre *lift* de precisão e taxa de falso positivo) em pelo
  menos 6 dos 8 casos automaticamente verificáveis do conjunto de teste.
- **RF-04:** para um `rule_id` inexistente ou vazio, o sistema retorna uma mensagem de erro
  clara, sem *stack trace* — verificável checando que a exceção é tratada e uma resposta
  estruturada de erro é devolvida.
- **RF-05:** para uma regra que não casa nenhuma transação na base, o sistema retorna uma
  resposta completa (precisão indefinida tratada como 0, sem erro de divisão por zero) —
  verificável deterministicamente.


# 7. Requisitos não funcionais e restrições

- **RNF-01:** latência mediana por chamada abaixo de 15 segundos.
- **RNF-02:** nenhuma chave de API é escrita neste notebook (carregada via variável de
  ambiente ou `getpass`).
- **RNF-03:** a parte determinística (métricas de backtest) é 100% reprodutível — mesma
  entrada produz sempre a mesma saída. O veredito qualitativo do LLM é estável mas não
  garantidamente idêntico *bit a bit* entre execuções, mesmo com `temperature=0` (limitação
  inerente a serviços de inferência distribuída).
- **RNF-04:** número de chamadas ao modelo e contagem de tokens de entrada/saída são
  registrados em toda execução.


# 8. Recursos externos potencialmente necessários

**Já disponíveis e usados neste baseline:**
- API da Groq (`langchain-groq`), para a única chamada de LLM do pipeline;
- pacote `riskops` (código-fonte deste mesmo grupo, Fase 1 do projeto): schema de regras,
  registro de regras versionado, avaliador vetorizado e métricas de backtest — obtido via
  clone raso do repositório público do projeto;
- amostra do dataset Bank Account Fraud (BAF), já incluída no repositório do projeto.

**Usados apenas na Fase 1 (não necessários para rodar este notebook):**
- API do Kaggle, para baixar o dataset completo (1.000.000 de linhas).

**Hipóteses para versões futuras (Entregáveis 2, 3, 4), não usadas aqui:**
- LangGraph (orquestração de múltiplos agentes);
- memória de longo prazo (ex.: grafo de conhecimento);
- LangSmith (observabilidade/tracing).


# 9. Tipo de baseline escolhido

**Classificação: parcial.**

1. **Por que é adequado:** o baseline resolve, de ponta a ponta, uma fatia representativa e
   diretamente útil do problema — avaliar uma regra por vez com evidência quantitativa — sem
   exigir nenhuma peça da arquitetura completa (orquestração, múltiplos agentes, memória).
2. **O que foi simplificado/excluído:** o monitoramento automático de todo o portfólio de
   regras (aqui o `rule_id` é escolhido manualmente); a geração de regras candidatas; o fluxo
   de aprovação humana; a publicação/versionamento de mudanças.
3. **Como permite comparação futura:** os Entregáveis seguintes podem envolver este mesmo
   diagnóstico em um laço que roda automaticamente sobre todas as regras ativas, ou acrescentar
   geração de candidatas — e medir, contra este mesmo conjunto de teste congelado, se a
   complexidade adicional melhora cobertura, precisão do veredito ou custo o suficiente para
   justificar o incremento.


# 10. Critérios preliminares de sucesso

| Critério | Requisito | Como será medido |
|---|---|---|
| Correção das métricas de backtest | RF-01 | verificação determinística (igualdade exata com `backtest_ruleset`) |
| Citação das métricas no parecer | RF-02 | verificação determinística (busca numérica com tolerância no texto) |
| Concordância do veredito | RF-03 | comparação com veredito de referência determinístico, meta ≥ 6/8 |
| Tratamento de erro (regra ausente/entrada vazia) | RF-04 | inspeção do retorno estruturado de erro |
| Robustez a regra sem correspondências | RF-05 | verificação determinística |
| Latência | RNF-01 | mediana das latências medidas nos 10 casos |
| Ausência de chave hardcoded | RNF-02 | inspeção manual do notebook antes da entrega |
| Reprodutibilidade da parte determinística | RNF-03 | reexecução do backtest, comparação exata |
| Custo/uso do modelo | RNF-04 | contagem de chamadas e tokens por execução |


# 11. Configuração do ambiente

Clonamos o repositório público do projeto (código-fonte da Fase 1: schema de regras,
registro versionado, avaliador e métricas de backtest, além da amostra do dataset e do
registro de regras já populado) e apontamos o interpretador diretamente para o código-fonte
clonado. **Nenhuma chave de API é escrita neste notebook** — ela é lida de uma variável de
ambiente já configurada ou solicitada de forma segura com `getpass`.


In [1]:
!git clone --depth 1 https://github.com/bronzattivinicius/riskops.git riskops_repo
%pip install -q -U langchain-groq pydantic pandas


Cloning into 'riskops_repo'...


remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (38/38), done.


remote: Total 50 (delta 5), reused 45 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 1.82 MiB | 2.17 MiB/s, done.
Resolving deltas: 100% (5/5), done.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# O pacote riskops eh Python puro (sem extensoes compiladas), entao basta apontar o
# interpretador para o codigo-fonte clonado -- evita a necessidade de reiniciar o kernel
# apos uma instalacao editavel via pip, uma limitacao conhecida do Python com pacotes
# ja carregados no mesmo processo.
import sys

sys.path.insert(0, "riskops_repo/src")

import riskops

print("riskops importado de:", riskops.__file__)


riskops importado de: /Users/viniciusbronzatti/Downloads/riskops/riskops_repo/src/riskops/__init__.py


In [3]:
import os, getpass, datetime, platform

# Este notebook nao usa observabilidade via LangSmith; desativa explicitamente para
# evitar tentativas de envio de traces (e possiveis erros de rede/autenticacao) caso a
# variavel LANGSMITH_TRACING esteja definida em algum .env local.
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

def carregar_chave_groq() -> str:
    """Funciona no Colab (userdata) e localmente (variável de ambiente)."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("INF0093-2026-2S")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)


Chave carregada via: variável de ambiente


In [4]:
from langchain_groq import ChatGroq

MODEL_NAME = "openai/gpt-oss-20b"
TEMPERATURE = 0
PROMPT_VERSAO = "v1"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Registro da execução: acompanha os resultados até o Entregável 4.
RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO


{'modelo': 'openai/gpt-oss-20b',
 'temperatura': 0,
 'prompt_versao': 'v1',
 'data': '2026-09-03T21:52:17',
 'python': '3.12.14'}

# 12. Dados ou entradas de exemplo

Carregamos a amostra do BAF (20.000 linhas, taxa de fraude preservada) e o registro de regras
já populado na Fase 1, ambos vindos do clone do repositório. O registro contém 8 regras reais
usadas no conjunto de teste: 3 regras "legadas" (ingênuas, nunca validadas estatisticamente) e
5 regras "candidatas" (3 fortes, 1 ambígua e 1 caso sintético de zero correspondências) — ver
detalhes na Seção 14.


In [5]:
from pathlib import Path
from riskops.data.loader import load_baf
from riskops.rules.store import RuleStore

REPO_DIR = Path("riskops_repo")
df = load_baf(path=REPO_DIR / "data" / "sample" / "baf_sample.csv")
store = RuleStore(root=REPO_DIR / "rule_registry")

print(f"{len(df)} transacoes carregadas | taxa de fraude na amostra: {df['fraud_bool'].mean():.2%}")
print(f"{len(store.list_rule_ids())} regras no registro:")
for rid in store.list_rule_ids():
    r = store.get(rid)
    print(f"  - {rid} ({r.status.value})")


20000 transacoes carregadas | taxa de fraude na amostra: 1.10%
8 regras no registro:
  - baf_device_email_reuse (candidate)
  - baf_foreign_request (active)
  - baf_free_email_domain (active)
  - baf_high_credit_risk_score (candidate)
  - baf_high_velocity_6h (active)
  - baf_impossible_threshold (candidate)
  - baf_invalid_phone_combo (candidate)
  - baf_weak_identity_match_elevated_risk (candidate)


# 13. Implementação do baseline

Pipeline linear, sem grafo e sem múltiplos agentes: dado um `rule_id`, (a) buscamos a regra no
registro, (b) rodamos o backtest determinístico dela contra a amostra do BAF (reaproveitando
100% o motor de regras da Fase 1), e (c) fazemos **uma única chamada de LLM** que interpreta as
métricas calculadas e devolve um parecer estruturado (Pydantic), o que torna o veredito
diretamente verificável em código.


In [6]:
import re
import time
from typing import Literal

from pydantic import BaseModel, Field

from riskops.metrics.backtest import backtest_ruleset
from riskops.rules.store import RuleNotFoundError


class RuleAssessment(BaseModel):
    """Parecer estruturado sobre uma regra de risco."""

    veredito: Literal["manter", "revisar", "aposentar"] = Field(
        description="Decisao recomendada para a regra, dadas as metricas de backtest."
    )
    justificativa: str = Field(
        description="Justificativa citando explicitamente os valores numericos das metricas."
    )
    sugestao: str = Field(
        description="Proximo passo concreto e acionavel."
    )


structured_llm = llm.with_structured_output(RuleAssessment, include_raw=True)

BASELINE_FRAUD_RATE = df["fraud_bool"].mean()


def veredito_referencia(metricas) -> str:
    """Veredito de referencia, calculado deterministicamente (sem LLM)."""
    if metricas.flagged_count == 0:
        return "aposentar"
    lift = (metricas.precision / BASELINE_FRAUD_RATE) if BASELINE_FRAUD_RATE else 0
    if lift >= 3 and metricas.false_positive_rate < 0.10:
        return "manter"
    if lift < 1.5 or metricas.false_positive_rate > 0.50:
        return "aposentar"
    return "revisar"


def montar_prompt(rule, metricas) -> str:
    return f"""Voce e um analista de risco avaliando uma regra de deteccao de fraude.

Regra: {rule.name} (id: {rule.id})
Descricao: {rule.description}
Status atual no registro: {rule.status.value}

Metricas de backtest contra {metricas.total} transacoes historicas (taxa de fraude na base: {BASELINE_FRAUD_RATE:.2%}):
- Taxa de deteccao de fraude (recall): {metricas.detection_rate:.1%}
- Taxa de falso positivo: {metricas.false_positive_rate:.1%}
- Precisao (fracao dos sinalizados que sao de fato fraude): {metricas.precision:.1%}
- Taxa de aprovacao (transacoes nao sinalizadas): {metricas.approval_rate:.1%}
- Total sinalizado: {metricas.flagged_count} de {metricas.total}

Com base apenas nessas metricas, de seu parecer: a regra deve ser MANTIDA, REVISADA ou
APOSENTADA? Justifique citando os numeros acima (arredondados) e sugira um proximo passo
concreto."""


def baseline(rule_id: str) -> dict:
    """Executa o diagnostico de uma regra de risco. Devolve um dict com resultado e instrumentacao."""
    inicio = time.perf_counter()

    if not rule_id:
        return {
            "ok": False, "erro": "rule_id vazio ou nao informado.",
            "latencia_s": round(time.perf_counter() - inicio, 2),
            "chamadas_llm": 0, "tokens_entrada": None, "tokens_saida": None,
        }

    try:
        rule = store.get(rule_id)
    except RuleNotFoundError:
        return {
            "ok": False, "erro": f"regra '{rule_id}' nao encontrada no registro.",
            "latencia_s": round(time.perf_counter() - inicio, 2),
            "chamadas_llm": 0, "tokens_entrada": None, "tokens_saida": None,
        }

    resultado_backtest = backtest_ruleset(df, [rule], label_col="fraud_bool")
    metricas = resultado_backtest.metrics

    prompt = montar_prompt(rule, metricas)
    saida = structured_llm.invoke(prompt)
    latencia = time.perf_counter() - inicio

    avaliacao = saida["parsed"]
    bruta = saida["raw"]
    uso = getattr(bruta, "usage_metadata", None) or {}

    if avaliacao is None:
        return {
            "ok": False, "erro": f"falha ao interpretar resposta do modelo: {saida.get('parsing_error')}",
            "latencia_s": round(latencia, 2), "chamadas_llm": 1,
            "tokens_entrada": uso.get("input_tokens"), "tokens_saida": uso.get("output_tokens"),
        }

    return {
        "ok": True,
        "veredito": avaliacao.veredito,
        "justificativa": avaliacao.justificativa,
        "sugestao": avaliacao.sugestao,
        "metricas_backtest": metricas,
        "veredito_referencia": veredito_referencia(metricas),
        "latencia_s": round(latencia, 2),
        "chamadas_llm": 1,
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
    }


# 14. Conjunto de avaliação

Conjunto **congelado**: os mesmos 10 casos serão executados nos Entregáveis 2, 3 e 4, usando
sempre a mesma amostra do BAF. Cobre regras normais (legadas fracas e candidatas fortes), um
caso ambíguo, um caso sintético sem nenhuma correspondência, e duas entradas inválidas.

| ID | rule_id | Tipo | Esperado |
|---|---|---|---|
| T01 | `baf_foreign_request` | normal (regra legada fraca) | revisar ou aposentar |
| T02 | `baf_free_email_domain` | normal (regra legada, ~ruído) | aposentar |
| T03 | `baf_high_velocity_6h` | normal (regra legada, pior que aleatório) | aposentar |
| T04 | `baf_high_credit_risk_score` | normal (candidata forte) | manter |
| T05 | `baf_weak_identity_match_elevated_risk` | normal (candidata forte nos dados completos) | manter ou revisar (baixo volume, sensível a ruído de amostragem) |
| T06 | `baf_device_email_reuse` | normal (candidata forte) | manter |
| T07 | `baf_invalid_phone_combo` | ambíguo | verificação manual |
| T08 | `baf_impossible_threshold` | zero correspondências | resposta graciosa, não crash |
| T09 | `"regra_fantasma"` (inexistente) | informação ausente | erro gracioso |
| T10 | `""` (vazio) | entrada incompleta | erro gracioso |


In [7]:
test_cases = [
    {"id": "T01", "rule_id": "baf_foreign_request", "tipo": "normal", "verificacao": "auto"},
    {"id": "T02", "rule_id": "baf_free_email_domain", "tipo": "normal", "verificacao": "auto"},
    {"id": "T03", "rule_id": "baf_high_velocity_6h", "tipo": "normal", "verificacao": "auto"},
    {"id": "T04", "rule_id": "baf_high_credit_risk_score", "tipo": "normal", "verificacao": "auto"},
    {"id": "T05", "rule_id": "baf_weak_identity_match_elevated_risk", "tipo": "normal", "verificacao": "auto"},
    {"id": "T06", "rule_id": "baf_device_email_reuse", "tipo": "normal", "verificacao": "auto"},
    {"id": "T07", "rule_id": "baf_invalid_phone_combo", "tipo": "ambiguo", "verificacao": "manual"},
    {"id": "T08", "rule_id": "baf_impossible_threshold", "tipo": "zero-match", "verificacao": "auto"},
    {"id": "T09", "rule_id": "regra_fantasma", "tipo": "informacao ausente", "verificacao": "auto"},
    {"id": "T10", "rule_id": "", "tipo": "entrada incompleta", "verificacao": "auto"},
]

print(len(test_cases), "casos definidos.")


10 casos definidos.


# 15. Implementação da verificação

Duas verificações determinísticas, correspondentes aos critérios da Seção 10: (a) o veredito
do modelo concorda com o veredito de referência calculado sem LLM (`veredito_referencia`,
definida na Seção 13); (b) a justificativa cita, com tolerância de arredondamento, os valores
numéricos das métricas calculadas. Para os casos de erro esperado (T09, T10), a verificação é
apenas checar que `ok=False` com uma mensagem de erro.


In [8]:
def metrica_citada(texto: str, valor_fracao: float, tolerancia_pp: float = 2.0) -> bool:
    """Verifica se um valor percentual aparece citado no texto, com tolerancia de arredondamento."""
    numeros = [float(n.replace(",", ".")) for n in re.findall(r"\d+[.,]?\d*", texto)]
    alvo = valor_fracao * 100
    return any(abs(n - alvo) <= tolerancia_pp for n in numeros)


def verificar_caso(caso: dict, resultado: dict) -> tuple:
    """Verifica um caso de teste executado. Devolve (aprovado_ou_None, observacao)."""
    if caso["verificacao"] == "manual":
        return None, "requer avaliacao manual (caso ambiguo)"

    if caso["tipo"] in ("informacao ausente", "entrada incompleta"):
        aprovado = not resultado["ok"] and bool(resultado.get("erro"))
        return aprovado, f"erro tratado: {resultado.get('erro')}"

    metricas = resultado["metricas_backtest"]
    ref = resultado["veredito_referencia"]
    veredito_ok = resultado["veredito"] == ref
    citacao_ok = metrica_citada(resultado["justificativa"], metricas.detection_rate) or \
        metrica_citada(resultado["justificativa"], metricas.false_positive_rate)
    aprovado = veredito_ok and citacao_ok
    obs = f"veredito={resultado['veredito']} referencia={ref} citacao_metrica={citacao_ok}"
    return aprovado, obs


# 16. Experimentos

Executamos o baseline sobre os 10 casos e registramos, para cada um: saída obtida, aprovação,
latência, tokens e observações.


In [9]:
registros = []

for caso in test_cases:
    resultado = baseline(caso["rule_id"])
    aprovado, observacao = verificar_caso(caso, resultado)
    registros.append({
        "id": caso["id"],
        "rule_id": caso["rule_id"],
        "tipo": caso["tipo"],
        "ok": resultado["ok"],
        "veredito": resultado.get("veredito"),
        "veredito_referencia": resultado.get("veredito_referencia"),
        "justificativa": resultado.get("justificativa"),
        "sugestao": resultado.get("sugestao"),
        "erro": resultado.get("erro"),
        "aprovado": aprovado,
        "observacao": observacao,
        "latencia_s": resultado["latencia_s"],
        "chamadas_llm": resultado["chamadas_llm"],
        "tokens_entrada": resultado["tokens_entrada"],
        "tokens_saida": resultado["tokens_saida"],
    })

print(len(registros), "execucoes registradas.")


10 execucoes registradas.


# 17. Resultados


In [10]:
import json

import pandas as pd

df_resultados = pd.DataFrame(registros)
display(df_resultados.drop(columns=["justificativa", "sugestao"]))

print()
for r in registros:
    if r["ok"]:
        print(f"[{r['id']}] {r['rule_id']}: {r['veredito']} (referencia: {r['veredito_referencia']})")
        print(f"   justificativa: {r['justificativa']}")
        print(f"   sugestao: {r['sugestao']}")
        print()

casos_auto = df_resultados[df_resultados["aprovado"].notna()]
RESUMO = {
    "taxa_aprovacao_casos_automaticos": round(casos_auto["aprovado"].mean(), 3) if len(casos_auto) else None,
    "casos_automaticos": int(len(casos_auto)),
    "casos_manuais": int(df_resultados["aprovado"].isna().sum()),
    "latencia_mediana_s": round(df_resultados["latencia_s"].median(), 2),
    "chamadas_llm_total": int(df_resultados["chamadas_llm"].sum()),
    "tokens_entrada_total": int(df_resultados["tokens_entrada"].dropna().sum()),
    "tokens_saida_total": int(df_resultados["tokens_saida"].dropna().sum()),
}
print(json.dumps(RESUMO, indent=2, ensure_ascii=False))

with open("baseline_v1_resultados.json", "w", encoding="utf-8") as f:
    json.dump({"run_info": RUN_INFO, "resumo": RESUMO, "registros": registros}, f, indent=2, ensure_ascii=False, default=str)


,id,rule_id,tipo,ok,veredito,veredito_referencia,erro,aprovado,observacao,latencia_s,chamadas_llm,tokens_entrada,tokens_saida
0,T01,baf_foreign_request,normal,True,revisar,revisar,NaN,True,veredito=revisar referencia=revisar citacao_me...,0.59,1,442.0,235.0
1,T02,baf_free_email_domain,normal,True,aposentar,aposentar,NaN,True,veredito=aposentar referencia=aposentar citaca...,0.80,1,461.0,382.0
2,T03,baf_high_velocity_6h,normal,True,revisar,aposentar,NaN,False,veredito=revisar referencia=aposentar citacao_...,0.51,1,480.0,238.0
3,T04,baf_high_credit_risk_score,normal,True,revisar,manter,NaN,False,veredito=revisar referencia=manter citacao_met...,0.71,1,444.0,260.0
4,T05,baf_weak_identity_match_elevated_risk,normal,True,revisar,revisar,NaN,True,veredito=revisar referencia=revisar citacao_me...,0.72,1,445.0,281.0
5,T06,baf_device_email_reuse,normal,True,revisar,manter,NaN,False,veredito=revisar referencia=manter citacao_met...,0.73,1,439.0,407.0
6,T07,baf_invalid_phone_combo,ambiguo,True,revisar,manter,NaN,None,requer avaliacao manual (caso ambiguo),0.70,1,450.0,272.0
7,T08,baf_impossible_threshold,zero-match,True,aposentar,aposentar,NaN,True,veredito=aposentar referencia=aposentar citaca...,0.72,1,442.0,262.0
8,T09,regra_fantasma,informacao ausente,False,NaN,NaN,regra 'regra_fantasma' nao encontrada no regis...,True,erro tratado: regra 'regra_fantasma' nao encon...,0.00,0,NaN,NaN
9,T10,,entrada incompleta,False,NaN,NaN,rule_id vazio ou nao informado.,True,erro tratado: rule_id vazio ou nao informado.,0.00,0,NaN,NaN



[T01] baf_foreign_request: revisar (referencia: revisar)
   justificativa: A regra apresenta recall de 6.8% e precisão de 3.1%, indicando que apenas 3.1% dos sinalizados são realmente fraudes, enquanto 93.9% são falsos positivos. O custo de sinalizar 487 transações de 20.000 (2.4% FP) é alto em relação ao ganho de 6.8% de recall, especialmente quando a taxa de fraude na base é 1.10%. O valor esperado de perdas evitadas é baixo comparado ao impacto operacional de falsos positivos.
   sugestao: Revisar a regra: introduzir um filtro adicional baseado em histórico de transações do cliente e/ou usar um modelo de risco ponderado por país, reduzindo falsos positivos e aumentando recall. Testar a nova lógica em um conjunto de validação antes de reativar.

[T02] baf_free_email_domain: aposentar (referencia: aposentar)
   justificativa: A regra flagga 52% das transações (10.490/20.000) mas apenas 1,4% (≈1%) dos sinalizados são realmente fraude, enquanto a taxa de fraude na base é 1,10%. O recal

**Interpretação:** a RF-01, RF-02, RF-04 e RF-05 foram plenamente atendidas. A RF-03 (meta:
concordância em pelo menos 6 dos 8 casos automáticos) **não foi atingida** -- ver a análise
completa na Seção 18, incluindo a causa mais provável (calibração do modelo, não erro de
leitura das métricas) e uma evidência concreta de não-determinismo entre execuções. A latência
mediana ficou bem dentro do limite da RNF-01 (15s).


# 18. Análise crítica do baseline

Preenchido com base na execução real registrada na Seção 17 (`RESUMO` e `registros`;
resultados salvos em `baseline_v1_resultados.json`).

1. **Requisitos plenamente atendidos:** RF-01 (métricas determinísticas), RF-04 (erro
   gracioso em T09/T10) e RF-05 (robustez ao caso sem correspondências, T08) foram atendidos
   em 100% dos casos -- todos vêm diretamente do motor de backtest da Fase 1, já coberto por
   28 testes unitários independentes. **RF-02 também foi plenamente atendido**: em todas as 8
   execuções bem-sucedidas, a justificativa citou os valores numéricos calculados
   (`citacao_metrica=True` em 8/8) -- o modelo nunca "inventou" um número.
2. **Requisito não atendido: RF-03.** A concordância do veredito com a referência
   determinística ficou em **2 de 8 (25%)** nesta execução -- bem abaixo da meta de 6/8. Isso
   **não é falha de leitura**: o modelo sempre citou os números certos (RF-02 em 100%), mas
   seu limiar de decisão diverge sistematicamente do nosso limiar de referência. Padrão
   observado: o modelo escolheu "revisar" em 6 dos 8 casos automáticos -- inclusive para
   regras claramente fortes (`baf_high_credit_risk_score`, `baf_device_email_reuse`, que a
   referência classifica como "manter") e para regras claramente ruins
   (`baf_free_email_domain`, `baf_high_velocity_6h`, que a referência classifica como
   "aposentar"). Só os dois casos mais extremos (o zero-match T08, e uma regra fraca em uma
   das execuções) receberam um veredito decisivo.
3. **Isso é uma limitação do modelo, da forma de medir, ou das duas?** Olhando a justificativa
   de T04 (`baf_high_credit_risk_score`, recall 5,9%, precisão 5,3%): o modelo argumenta que
   "a baixa precisão indica que a maioria dos alertas não são fraudes" e recomenda revisar,
   enquanto nossa referência (limiar simples de *lift* >= 3x) diz "manter". O modelo está
   aplicando um padrão de decisão mais rigoroso e plausível de negócio (94,7% de falso alarme
   entre os sinalizados é um custo operacional real) do que nosso limiar simplificado --
   ou seja, parte da "reprovação" da RF-03 pode ser um limiar de referência simplista demais,
   não um erro de julgamento do modelo. Isso é exatamente o tipo de limitação da forma de
   medir, e não do sistema, que o enunciado pede para identificar.
4. **Reprodutibilidade (RNF-03), evidência concreta:** rodamos o baseline duas vezes, mesma
   configuração (`temperature=0`, mesmo prompt, mesmos dados). O caso T01
   (`baf_foreign_request`) recebeu vereditos **diferentes** nas duas execuções ("revisar" na
   primeira, "aposentar" na segunda) -- confirmando na prática o alerta do próprio enunciado de
   que `temperature=0` reduz mas não elimina a variação em serviços de inferência distribuída.
5. **Latência (RNF-01):** mediana de 0,64s por chamada -- muito abaixo do limite de 15s.
   A Groq é rápida o suficiente para que latência não seja, hoje, um fator limitante.
6. **Entradas mais difíceis:** os casos com sinal "mediano" (recall razoável, precisão baixa
   -- T04, T06, T07) são consistentemente os mais difíceis, tanto para o modelo quanto para o
   nosso próprio limiar de referência -- é exatamente onde uma decisão de negócio genuína
   entra em jogo, e onde uma verificação puramente determinística é mais frágil.
7. **Limitações do modelo vs. da arquitetura:** a divergência de calibração do veredito
   (item 2-3 acima) é uma limitação do **modelo/prompt** -- poderia mudar trocando o modelo,
   ajustando o prompt, ou dando exemplos calibrados (few-shot). Já a incapacidade de comparar
   uma regra contra as outras 7 do registro, ou de lembrar decisões passadas sobre a mesma
   regra, é uma limitação da **arquitetura** (pipeline de uma regra por vez, sem memória) --
   não seria resolvida trocando o modelo, só adicionando estrutura (ver Seção 19).
8. **Ruído de amostragem, não do sistema:** `baf_weak_identity_match_elevated_risk` é uma
   regra claramente forte na base completa de 1.000.000 de linhas (4,87% de taxa de fraude,
   6,6x a base), mas a referência calculada sobre a amostra de 20.000 linhas oscilou entre
   "manter" e "revisar" dependendo do subconjunto exato de fraudes sorteadas -- o volume que
   ela sinaliza é pequeno o suficiente para a estimativa de precisão ficar sensível a poucos
   casos a mais ou a menos. Isso é uma limitação da forma de medir (tamanho da amostra), não
   da regra nem da arquitetura -- o motor completo (1M de linhas) não sofreria disso.


# 19. Possíveis evoluções arquiteturais

- **Paralelização (votação/autoconsistência):** diretamente justificada pela limitação
  observada na Seção 18, item 4 -- o mesmo caso (T01) recebeu vereditos diferentes em duas
  execuções idênticas (`temperature=0`). Rodar a mesma chamada 3-5 vezes e tomar o veredito
  majoritário atacaria exatamente essa instabilidade, ao custo de 3-5x mais chamadas/latência
  por regra avaliada -- um custo mensurável que pode ser comparado contra o ganho de
  concordância na RF-03.
- **Workflow (pipeline com múltiplas etapas fixas):** justificado se quisermos rodar este
  mesmo diagnóstico automaticamente sobre *todas* as regras ativas do registro (hoje é preciso
  informar um `rule_id` manualmente) -- resolveria a limitação de escopo "uma regra por vez".
  Custo: mais chamadas ao modelo (uma por regra), mais latência agregada.
- **Calibração do prompt (few-shot):** justificada pela limitação observada na Seção 18, item
  2-3 -- o modelo tem um limiar de decisão diferente do nosso limiar de referência simples.
  Incluir 2-3 exemplos rotulados manualmente (regra + métricas + veredito esperado, com
  justificativa do porquê) no prompt poderia calibrar melhor o modelo sem mudar a arquitetura.
  Vale tentar antes de adicionar complexidade estrutural.
- **ReAct / ferramentas:** não claramente justificado ainda neste estágio -- o baseline já tem
  acesso direto e completo às métricas via Python, sem necessidade de o modelo decidir
  dinamicamente que ferramenta chamar. Só faria sentido se o sistema precisasse buscar dados
  adicionais (ex.: histórico de mudanças da regra) que não estão disponíveis de antemão.
- **Memória:** justificada quando o sistema precisar lembrar de diagnósticos anteriores da
  mesma regra (ex.: "essa regra já foi sinalizada para revisão há dois meses e nada mudou") --
  não resolvida pelo baseline atual, que trata cada chamada de forma independente.
- **Ferramentas/MCP:** não justificado agora; poderia entrar se precisássemos consultar
  sistemas externos (ex.: um MCP de ticket/aprovação) para de fato registrar a decisão.
- **Planejamento:** não justificado para uma decisão de um único passo como esta.
- **Múltiplos agentes:** justificado apenas quando o sistema precisar *gerar* regras
  candidatas (papel distinto de *avaliar* uma regra existente) e/ou envolver aprovação humana
  como uma etapa formal do fluxo -- exatamente o Analista/Gerador de Regras/Revisor/Gerente de
  Risco da arquitetura-alvo do RiskOps (ver `README.md`). Introduzir isso agora, sem que o
  baseline de uma-regra-por-vez tenha demonstrado essa necessidade, seria complexidade não
  justificada -- por isso não faz parte deste Entregável 1.


# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final apresenta
> vantagens em relação ao baseline?**

Hipótese inicial: a arquitetura final (grafo multiagente do RiskOps) só se justifica se, sobre
o **mesmo conjunto de 10 casos de teste congelados** e a **mesma amostra do BAF**, ela igualar
ou superar o baseline nas métricas já instrumentadas aqui (taxa de concordância do veredito
com a referência, cobertura dos requisitos RF/RNF) **e**, adicionalmente, resolver ao menos uma
limitação explicitamente identificada na Seção 18 que o baseline não resolve por construção --
por exemplo, avaliar o portfólio inteiro de regras automaticamente (não apenas uma por vez),
gerar e comparar regras candidatas novas, ou manter histórico de decisões entre execuções.

Evidências e métricas que usaremos: (i) as mesmas métricas de backtest (`detection_rate`,
`false_positive_rate`, `precision`, `approval_rate`) para qualquer regra nova gerada pela
arquitetura final, comparadas contra as regras legadas via `compare_rulesets` (já implementado
na Fase 1); (ii) a taxa de concordância do veredito com a referência determinística, no mesmo
conjunto de 10 casos -- **o baseline atual atingiu 25% (2/8) nesta execução**, então qualquer
evolução (calibração de prompt, votação por autoconsistência, ou arquitetura multiagente) só
será considerada uma melhoria real se elevar essa taxa de forma mensurável, não apenas parecer
mais sofisticada; (iii) latência e custo (tokens/chamadas) agregados, para quantificar o preço
da complexidade adicional -- uma arquitetura mais cara que não melhora as métricas anteriores
não seria considerada uma evolução justificada.


# 21. Conclusão

Este Entregável 1 implementa o primeiro passo do projeto RiskOps: um baseline **parcial**,
não multiagente, que combina o motor de regras determinístico construído na Fase 1 (schema de
regras, registro versionado e auditável, avaliador vetorizado, métricas de backtest -- todos
já testados por 28 testes unitários) com **uma única chamada de LLM** (Groq/`openai/gpt-oss-20b`)
que interpreta essas métricas e produz um parecer estruturado e verificável sobre se uma regra
de risco deve ser mantida, revisada ou aposentada.

Os resultados reais sobre os 10 casos de teste congelados (Seção 17) mostram um baseline que
atende integralmente RF-01, RF-02, RF-04 e RF-05, mas **não atinge a meta da RF-03** (25-37,5%
de concordância do veredito com a referência determinística, em duas execuções, contra uma meta
de 75%). A análise crítica (Seção 18) aponta a causa mais provável: o modelo tende a responder
"revisar" mesmo diante de sinais claros, e apresentamos evidência concreta de que o próprio
veredito varia entre execuções idênticas -- uma limitação real, medida, não hipotética. Isso
aponta caminhos concretos e priorizados para os próximos entregáveis (calibração de prompt,
votação por autoconsistência, e só então arquitetura multiagente), em vez de justificar
complexidade adicional genérica. A hipótese para a próxima versão (Seção 20) é usar exatamente
este mesmo baseline, conjunto de teste e métricas como a régua de comparação para qualquer
incremento de arquitetura proposto daqui em diante.


---

# Checklist antes da entrega

- [x] O problema está claramente definido.
- [x] O usuário-alvo foi identificado.
- [x] Escopo e não-objetivos estão explícitos.
- [x] Existem requisitos funcionais **verificáveis** (RF-01 a RF-05).
- [x] Existem requisitos não funcionais (RNF-01 a RNF-04).
- [x] Cada critério de sucesso diz **como** será medido (Seção 10).
- [x] O tipo de baseline foi classificado (parcial) e justificado (Seção 9).
- [x] O baseline executa sem erros (executado de ponta a ponta com a Groq real).
- [x] Existem dez casos de teste, cobrindo mais de um tipo (normal, ambíguo, zero-match, entrada ausente/incompleta).
- [x] Modelo, temperatura, data, latências e tokens estão registrados (`RUN_INFO`, `baseline_v1_resultados.json`).
- [x] Notebook executado do início ao fim, com saídas visíveis, sem chave de API escrita nele.
